# Breakout DQN
**Algorithm:** Deep Q-Network  
**Training target:** 15,000,000 timesteps  
**Checkpoint:** every 1,000,000 timesteps  
**Resume:** automatically continues from the latest checkpoint

## 1. Setup
Install the required Atari and reinforcement learning packages.

In [ ]:
%pip install -q \
    "gymnasium[atari]>=1.0" \
    "stable-baselines3[extra]>=2.4" \
    "ale-py>=0.10" \
    "opencv-python-headless" \
    "imageio[ffmpeg]" \
    "pandas" \
    "matplotlib"

## 2. Configuration
Set the seed, folders, evaluation episodes, and device.

In [ ]:
import os
os.environ.setdefault(
    "TF_CPP_MIN_LOG_LEVEL",
    "3",
)
os.environ.setdefault(
    "TF_ENABLE_ONEDNN_OPTS",
    "0",
)

import re
import time
import random
import base64
import contextlib
import sys
import io
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

import torch as th
import torch.nn as nn

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecVideoRecorder
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.utils import set_random_seed
from IPython import display as ipythondisplay

SEED = 42
EVAL_EPISODES = 20

random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
set_random_seed(SEED)

DEVICE = (
    "cuda"
    if th.cuda.is_available()
    else "cpu"
)

ROOT = Path("as2_breakout")
MODEL_DIR = ROOT / "models"
CHECKPOINT_DIR = ROOT / "checkpoints"
LOG_DIR = ROOT / "logs"
VIDEO_DIR = ROOT / "videos"

DQN_CHECKPOINT_DIR = (
    CHECKPOINT_DIR
    / "dqn"
)

DQN_LOG_PATH = (
    LOG_DIR
    / "dqn_training_log.csv"
)

for folder in [
    MODEL_DIR,
    DQN_CHECKPOINT_DIR,
    LOG_DIR,
    VIDEO_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Device:", DEVICE)

## 3. Training Target
Training can be extended by increasing `TOTAL_TIMESTEPS`. Training statistics are written to CSV every 100,000 timesteps.

In [ ]:
TOTAL_TIMESTEPS = 15_000_000
CHECKPOINT_INTERVAL_STEPS = 1_000_000
LOG_INTERVAL_STEPS = 100_000

## 4. Fixed Hyperparameters
DQN hyperparameters are kept in one place for reproducibility and tuning.

In [ ]:
DQN_LEARNING_RATE = 5e-5
DQN_N_ENVS = 1
DQN_BUFFER_SIZE = 150_000
DQN_LEARNING_STARTS = 20_000
DQN_BATCH_SIZE = 32
DQN_GAMMA = 0.99
DQN_TRAIN_FREQ = 4
DQN_GRADIENT_STEPS = 1
DQN_TARGET_UPDATE_INTERVAL = 10_000
DQN_EXPLORATION_FRACTION = 0.10
DQN_EXPLORATION_INITIAL_EPS = 1.0
DQN_EXPLORATION_FINAL_EPS = 0.05

CNN_FEATURES_DIM = 512

## 5. Environment
Breakout uses stacked image observations and discrete actions. The objective is to maximise cumulative reward.

In [ ]:
ENV_ID = "ale_py:ALE/Breakout-v5"


@contextlib.contextmanager
def suppress_stderr():
    try:
        stderr_fd = sys.stderr.fileno()
    except (AttributeError, io.UnsupportedOperation):
        yield
        return

    saved_fd = os.dup(stderr_fd)

    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(
                devnull.fileno(),
                stderr_fd,
            )
            yield
    finally:
        os.dup2(
            saved_fd,
            stderr_fd,
        )
        os.close(saved_fd)


def make_breakout_env(
    seed,
    render_mode=None,
):
    def _init():
        with suppress_stderr():
            env = gym.make(
                ENV_ID,
                frameskip=1,
                repeat_action_probability=0.0,
                render_mode=render_mode,
            )

        env = AtariWrapper(
            env,
            screen_size=84,
            frame_skip=4,
            terminal_on_life_loss=False,
            clip_reward=False,
        )

        env = Monitor(env)
        env.reset(seed=seed)
        return env

    return _init


def build_env(
    seed=SEED,
    n_envs=1,
    render_mode=None,
):
    env = DummyVecEnv([
        make_breakout_env(
            seed + i,
            render_mode=render_mode,
        )
        for i in range(n_envs)
    ])

    return VecFrameStack(
        env,
        n_stack=4,
    )


print("Environment:", ENV_ID)

## 6. Custom CNN
A custom CNN extracts visual features from stacked Atari observations.

In [ ]:
class CustomCNN(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space,
        features_dim=CNN_FEATURES_DIM,
    ):
        super().__init__(
            observation_space,
            features_dim,
        )

        channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(
                channels,
                32,
                kernel_size=8,
                stride=4,
            ),
            nn.ReLU(),
            nn.Conv2d(
                32,
                64,
                kernel_size=4,
                stride=2,
            ),
            nn.ReLU(),
            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                stride=1,
            ),
            nn.ReLU(),
            nn.Flatten(),
        )

        with th.no_grad():
            sample = th.as_tensor(
                observation_space.sample()[None]
            ).float()

            n_flatten = self.cnn(
                sample
            ).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(
                n_flatten,
                features_dim,
            ),
            nn.ReLU(),
        )

    def forward(
        self,
        observations,
    ):
        return self.linear(
            self.cnn(observations)
        )


POLICY_KWARGS = {
    "features_extractor_class": CustomCNN,
    "features_extractor_kwargs": {
        "features_dim": CNN_FEATURES_DIM,
    },
}

## 7. Utilities
Training logs, checkpoints, evaluation, and plotting are handled by reusable utilities.

In [ ]:
class TrainingLogCallback(BaseCallback):
    def __init__(
        self,
        log_path,
        algorithm,
        log_interval=100_000,
    ):
        super().__init__()

        self.log_path = Path(log_path)
        self.algorithm = algorithm
        self.log_interval = int(
            log_interval
        )

        self.next_log = None
        self.interval_start = None
        self.recent_rewards = []
        self.recent_lengths = []

        self.log_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    def _on_training_start(self):
        current = int(
            self.model.num_timesteps
        )

        self.interval_start = (
            current
            // self.log_interval
        ) * self.log_interval

        self.next_log = (
            self.interval_start
            + self.log_interval
        )

    def _write_row(
        self,
        row,
    ):
        new_row = pd.DataFrame(
            [row]
        )

        if self.log_path.exists():
            existing = pd.read_csv(
                self.log_path
            )

            new_row = pd.concat(
                [
                    existing,
                    new_row,
                ],
                ignore_index=True,
            )

            new_row = (
                new_row
                .drop_duplicates(
                    subset=["timestep"],
                    keep="last",
                )
                .sort_values(
                    "timestep"
                )
            )

        new_row.to_csv(
            self.log_path,
            index=False,
        )

    def _on_step(self):
        for info in self.locals.get(
            "infos",
            [],
        ):
            if "episode" in info:
                self.recent_rewards.append(
                    float(
                        info["episode"]["r"]
                    )
                )

                self.recent_lengths.append(
                    int(
                        info["episode"]["l"]
                    )
                )

        current = int(
            self.model.num_timesteps
        )

        if current < self.next_log:
            return True

        if self.recent_rewards:
            episodes = len(
                self.recent_rewards
            )

            mean_reward = float(
                np.mean(
                    self.recent_rewards
                )
            )

            mean_length = float(
                np.mean(
                    self.recent_lengths
                )
            )

        else:
            episodes = 0
            mean_reward = np.nan
            mean_length = np.nan

        exploration = getattr(
            self.model,
            "exploration_rate",
            np.nan,
        )

        row = {
            "algorithm": self.algorithm,
            "timestep": current,
            "interval_start": self.interval_start,
            "interval_end": self.next_log,
            "episodes": episodes,
            "ep_rew_mean": mean_reward,
            "ep_len_mean": mean_length,
            "exploration_rate": exploration,
        }

        self._write_row(row)

        line = (
            f"timestep={current:,} | "
            f"interval="
            f"{self.interval_start:,}-"
            f"{self.next_log:,} | "
            f"episodes={episodes} | "
            f"ep_rew_mean="
            f"{mean_reward:.2f} | "
            f"ep_len_mean="
            f"{mean_length:.1f}"
        )

        if not np.isnan(
            exploration
        ):
            line += (
                " | exploration_rate="
                f"{exploration:.3f}"
            )

        print(line)

        self.recent_rewards.clear()
        self.recent_lengths.clear()

        self.interval_start = (
            self.next_log
        )

        while (
            self.next_log
            <= current
        ):
            self.next_log += (
                self.log_interval
            )

        return True


class SaveOnIntervalCallback(BaseCallback):
    def __init__(
        self,
        save_interval,
        save_path,
        prefix,
        save_replay_buffer=False,
    ):
        super().__init__()

        self.save_interval = int(
            save_interval
        )

        self.save_path = Path(
            save_path
        )

        self.prefix = prefix
        self.save_replay_buffer = (
            save_replay_buffer
        )

        self.next_save = None

        self.save_path.mkdir(
            parents=True,
            exist_ok=True,
        )

    def _on_training_start(self):
        current = int(
            self.model.num_timesteps
        )

        self.next_save = (
            (
                current
                // self.save_interval
            )
            + 1
        ) * self.save_interval

    def _save_checkpoint(self):
        steps = int(
            self.model.num_timesteps
        )

        model_path = (
            self.save_path
            / (
                f"{self.prefix}_"
                f"{steps}_steps"
            )
        )

        self.model.save(
            model_path
        )

        if (
            self.save_replay_buffer
            and hasattr(
                self.model,
                "save_replay_buffer",
            )
        ):
            replay_path = (
                self.save_path
                / (
                    f"{self.prefix}_"
                    "latest_replay_buffer.pkl"
                )
            )

            self.model.save_replay_buffer(
                replay_path
            )

            step_path = (
                self.save_path
                / (
                    f"{self.prefix}_"
                    "latest_replay_buffer_steps.txt"
                )
            )

            step_path.write_text(
                str(steps),
                encoding="utf-8",
            )

        print(
            "Checkpoint saved:",
            str(model_path) + ".zip",
        )

    def _on_step(self):
        current = int(
            self.model.num_timesteps
        )

        if current >= self.next_save:
            self._save_checkpoint()

            while (
                self.next_save
                <= current
            ):
                self.next_save += (
                    self.save_interval
                )

        return True


def latest_step_checkpoint(
    folder,
    prefix,
):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_steps.zip"
        )
    )

    if not files:
        return None, 0

    def get_steps(path):
        match = re.search(
            r"_(\d+)_steps\.zip$",
            path.name,
        )

        return (
            int(match.group(1))
            if match
            else -1
        )

    latest = max(
        files,
        key=get_steps,
    )

    return (
        latest,
        get_steps(latest),
    )


def load_training_log(
    path,
):
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    return (
        pd.read_csv(path)
        .drop_duplicates(
            subset=["timestep"],
            keep="last",
        )
        .sort_values(
            "timestep"
        )
        .reset_index(
            drop=True
        )
    )


def plot_learning_curve(
    log_path,
    title,
    smoothing_points=5,
):
    frame = load_training_log(
        log_path
    )

    if frame.empty:
        print(
            "No saved training log found."
        )
        return frame

    frame["smoothed_reward"] = (
        frame["ep_rew_mean"]
        .rolling(
            smoothing_points,
            min_periods=1,
        )
        .mean()
    )

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        frame["timestep"],
        frame["ep_rew_mean"],
        alpha=0.35,
        label="100k interval mean",
    )

    plt.plot(
        frame["timestep"],
        frame["smoothed_reward"],
        linewidth=2,
        label=(
            f"{smoothing_points}-point "
            "moving mean"
        ),
    )

    plt.title(title)
    plt.xlabel("Timesteps")
    plt.ylabel(
        "Mean Episode Reward"
    )
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    return frame


def evaluate_checkpoint(
    model_class,
    checkpoint,
    episodes=EVAL_EPISODES,
    custom_objects=None,
):
    model = model_class.load(
        checkpoint,
        device=DEVICE,
        custom_objects=custom_objects,
    )

    env = build_env(
        SEED + 1000,
        n_envs=1,
    )

    rewards, lengths = evaluate_policy(
        model,
        env,
        n_eval_episodes=episodes,
        deterministic=True,
        return_episode_rewards=True,
        warn=False,
    )

    env.close()

    result = {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }

    del model
    del env

    gc.collect()

    if th.cuda.is_available():
        th.cuda.empty_cache()

    return result


def evaluate_all_checkpoints(
    model_class,
    folder,
    prefix,
    custom_objects=None,
):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_steps.zip"
        )
    )

    if not files:
        return pd.DataFrame()

    def get_steps(path):
        match = re.search(
            r"_(\d+)_steps\.zip$",
            path.name,
        )

        return int(
            match.group(1)
        )

    files = sorted(
        files,
        key=get_steps,
    )

    rows = []

    for checkpoint in files:
        steps = get_steps(
            checkpoint
        )

        print(
            f"Evaluating "
            f"{steps:,} steps..."
        )

        result = evaluate_checkpoint(
            model_class,
            checkpoint,
            custom_objects=custom_objects,
        )

        rows.append({
            "timestep": steps,
            **result,
        })

    return pd.DataFrame(
        rows
    )


def random_baseline(
    episodes=EVAL_EPISODES,
):
    env = build_env(
        SEED + 99,
        n_envs=1,
    )

    rewards = []
    lengths = []

    obs = env.reset()
    total_reward = 0.0
    total_length = 0

    while len(rewards) < episodes:
        action = np.array([
            env.action_space.sample()
        ])

        obs, reward, done, info = (
            env.step(action)
        )

        total_reward += float(
            reward[0]
        )

        total_length += 1

        if bool(done[0]):
            rewards.append(
                total_reward
            )

            lengths.append(
                total_length
            )

            total_reward = 0.0
            total_length = 0

    env.close()

    return {
        "mean_reward": float(
            np.mean(rewards)
        ),
        "std_reward": float(
            np.std(rewards)
        ),
        "median_reward": float(
            np.median(rewards)
        ),
        "mean_length": float(
            np.mean(lengths)
        ),
        "max_reward": float(
            np.max(rewards)
        ),
    }

## 8. Random Baseline
Evaluate a random policy before training.

In [ ]:
random_result = (
    random_baseline()
)

random_result

## 9. DQN Training
DQN resumes from the latest checkpoint when one exists. The latest replay buffer is restored only when its timestep matches the model checkpoint.

In [ ]:
dqn_env = build_env(
    SEED,
    n_envs=DQN_N_ENVS,
)

dqn_ckpt, completed_steps = (
    latest_step_checkpoint(
        DQN_CHECKPOINT_DIR,
        "breakout_dqn",
    )
)

if dqn_ckpt is None:
    print(
        "No DQN checkpoint found. "
        "Starting from 0."
    )

    dqn = DQN(
        "CnnPolicy",
        dqn_env,
        learning_rate=DQN_LEARNING_RATE,
        buffer_size=DQN_BUFFER_SIZE,
        learning_starts=DQN_LEARNING_STARTS,
        batch_size=DQN_BATCH_SIZE,
        gamma=DQN_GAMMA,
        train_freq=DQN_TRAIN_FREQ,
        gradient_steps=DQN_GRADIENT_STEPS,
        target_update_interval=DQN_TARGET_UPDATE_INTERVAL,
        exploration_fraction=DQN_EXPLORATION_FRACTION,
        exploration_initial_eps=DQN_EXPLORATION_INITIAL_EPS,
        exploration_final_eps=DQN_EXPLORATION_FINAL_EPS,
        policy_kwargs=POLICY_KWARGS,
        verbose=0,
        seed=SEED,
        device=DEVICE,
    )

else:
    print(
        "Resuming DQN from:",
        dqn_ckpt,
    )

    dqn = DQN.load(
        dqn_ckpt,
        env=dqn_env,
        device=DEVICE,
        custom_objects={
            "buffer_size": DQN_BUFFER_SIZE,
        },
    )

    replay_path = (
        DQN_CHECKPOINT_DIR
        / (
            "breakout_dqn_"
            "latest_replay_buffer.pkl"
        )
    )

    replay_steps_path = (
        DQN_CHECKPOINT_DIR
        / (
            "breakout_dqn_"
            "latest_replay_buffer_steps.txt"
        )
    )

    replay_loaded = False

    if (
        replay_path.exists()
        and replay_steps_path.exists()
    ):
        replay_steps = int(
            replay_steps_path
            .read_text(
                encoding="utf-8"
            )
            .strip()
        )

        if (
            replay_steps
            == completed_steps
        ):
            try:
                dqn.load_replay_buffer(
                    replay_path
                )
                replay_loaded = True
            except Exception as error:
                print(
                    "Replay buffer not loaded:",
                    error,
                )

    print(
        "Replay buffer loaded:",
        replay_loaded,
    )

log_callback = TrainingLogCallback(
    DQN_LOG_PATH,
    "DQN",
    LOG_INTERVAL_STEPS,
)

save_callback = SaveOnIntervalCallback(
    CHECKPOINT_INTERVAL_STEPS,
    DQN_CHECKPOINT_DIR,
    "breakout_dqn",
    save_replay_buffer=True,
)

remaining_steps = max(
    0,
    TOTAL_TIMESTEPS
    - int(
        dqn.num_timesteps
    ),
)

print(
    "Target steps:",
    TOTAL_TIMESTEPS,
)
print(
    "Current steps:",
    dqn.num_timesteps,
)
print(
    "Remaining steps:",
    remaining_steps,
)

if remaining_steps > 0:
    start = time.time()

    dqn.learn(
        total_timesteps=remaining_steps,
        reset_num_timesteps=False,
        callback=CallbackList([
            log_callback,
            save_callback,
        ]),
        progress_bar=True,
    )

    print(
        "Training time:",
        round(
            (
                time.time()
                - start
            )
            / 60,
            1,
        ),
        "minutes",
    )

dqn.save(
    MODEL_DIR
    / "breakout_dqn_final"
)

## 10. Learning Curve
The curve is read from the persistent CSV log, so it remains available after the notebook is restarted.

In [ ]:
dqn_curve = plot_learning_curve(
    DQN_LOG_PATH,
    "Breakout DQN Learning Curve",
    smoothing_points=5,
)

display(
    dqn_curve.tail(10)
)

## 11. Checkpoint Evaluation
Every saved checkpoint is evaluated with deterministic actions. The checkpoint with the highest mean reward is selected.

In [ ]:
dqn_results = (
    evaluate_all_checkpoints(
        DQN,
        DQN_CHECKPOINT_DIR,
        "breakout_dqn",
        custom_objects={
            "buffer_size": 1,
        },
    )
)

display(
    dqn_results
)

best_dqn = (
    dqn_results
    .sort_values(
        "mean_reward",
        ascending=False,
    )
    .iloc[0]
)

print(
    "BEST CHECKPOINT"
)
print(
    "Timestep:",
    f"{int(best_dqn['timestep']):,}",
)
print(
    "Mean reward:",
    round(
        best_dqn[
            "mean_reward"
        ],
        2,
    ),
)
print(
    "Std reward:",
    round(
        best_dqn[
            "std_reward"
        ],
        2,
    ),
)
print(
    "Median reward:",
    round(
        best_dqn[
            "median_reward"
        ],
        2,
    ),
)
print(
    "Mean length:",
    round(
        best_dqn[
            "mean_length"
        ],
        2,
    ),
)
print(
    "Max reward:",
    round(
        best_dqn[
            "max_reward"
        ],
        2,
    ),
)

## 12. Results
Compare the random baseline with the best DQN checkpoint.

In [ ]:
dqn_summary = pd.DataFrame([
    {
        "Algorithm": "Random",
        **random_result,
    },
    {
        "Algorithm": "DQN",
        "mean_reward": best_dqn[
            "mean_reward"
        ],
        "std_reward": best_dqn[
            "std_reward"
        ],
        "median_reward": best_dqn[
            "median_reward"
        ],
        "mean_length": best_dqn[
            "mean_length"
        ],
        "max_reward": best_dqn[
            "max_reward"
        ],
    },
])

dqn_summary

## 13. Record and Play Video
Record the best DQN checkpoint and play the resulting video directly in the notebook.

In [ ]:
def show_videos(
    video_path=VIDEO_DIR,
    prefix="",
):
    html = []

    for mp4 in sorted(
        Path(video_path).glob(
            f"{prefix}*.mp4"
        )
    ):
        video_b64 = (
            base64.b64encode(
                mp4.read_bytes()
            )
            .decode("ascii")
        )

        html.append(
            f"""
            <video controls
                   style="height: 400px;
                          max-width: 100%;">
                <source
                    src="data:video/mp4;base64,{video_b64}"
                    type="video/mp4">
            </video>
            """
        )

    if not html:
        print(
            "No video found for prefix:",
            prefix,
        )
        return

    ipythondisplay.display(
        ipythondisplay.HTML(
            data="<br>".join(html)
        )
    )


def record_video(
    model,
    prefix,
    video_length=3000,
    seed=SEED + 5000,
):
    env = build_env(
        seed=seed,
        n_envs=1,
        render_mode="rgb_array",
    )

    env = VecVideoRecorder(
        env,
        video_folder=str(
            VIDEO_DIR
        ),
        record_video_trigger=(
            lambda step: step == 0
        ),
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = env.reset()

    for _ in range(
        video_length
    ):
        action, _ = model.predict(
            obs,
            deterministic=True,
        )

        obs, _, _, _ = env.step(
            action
        )

    env.close()

In [ ]:
best_steps = int(
    best_dqn["timestep"]
)

best_checkpoint = (
    DQN_CHECKPOINT_DIR
    / (
        "breakout_dqn_"
        f"{best_steps}_steps.zip"
    )
)

video_model = DQN.load(
    best_checkpoint,
    device=DEVICE,
    custom_objects={
        "buffer_size": 1,
    },
)

record_video(
    video_model,
    prefix=(
        "breakout-dqn-best"
    ),
)

show_videos(
    VIDEO_DIR,
    prefix=(
        "breakout-dqn-best"
    ),
)

del video_model

gc.collect()

if th.cuda.is_available():
    th.cuda.empty_cache()